# 00 · Môi trường và dữ liệu

Chạy notebook này **một lần duy nhất** khi bắt đầu. Nó tải Kvasir-SEG, chia
train/val/test cố định, và tính tỉ lệ mất cân bằng lớp — con số dùng cho
mục Dữ liệu trong báo cáo.

Người phụ trách: **SV B**.

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

## 1. Tải dữ liệu

Bỏ qua ô này nếu `data/Kvasir-SEG` đã có.

In [ ]:
from src.data import KVASIR_URL

if not Path("data/Kvasir-SEG/images").is_dir():
    !wget -q {KVASIR_URL} -O kvasir-seg.zip
    !mkdir -p data && unzip -q kvasir-seg.zip -d data/ && rm kvasir-seg.zip

n_images = len(list(Path("data/Kvasir-SEG/images").glob("*.jpg")))
print(f"Số ảnh: {n_images}")

## 2. Chia tập cố định

Chia **một lần** rồi ghi ra `splits/*.txt` và commit vào git. Không bao giờ
chia lại ngẫu nhiên, nếu không mọi so sánh giữa các cấu hình sẽ vô nghĩa.

In [ ]:
cfg = Config()
splits = make_splits(cfg.data_root, cfg.split_dir, seed=cfg.seed)
for name, names in splits.items():
    print(f"{name:5s}: {len(names):4d} ảnh")

## 3. Mức mất cân bằng lớp

Đây là lý do tồn tại của cả nghiên cứu về hàm mất mát. Đưa con số này vào
báo cáo, và dùng `pos_weight` cho Weighted BCE.

In [ ]:
ratio = foreground_ratio(cfg.data_root, splits["train"], sample=None)
pw = pos_weight_from_ratio(ratio)
print(f"Tỉ lệ pixel polyp : {ratio:.4f}  ({ratio*100:.2f}%)")
print(f"pos_weight        : {pw:.2f}")

## 3b. Đường cơ sở tầm thường

Nếu một mô hình dự đoán TOÀN BỘ ảnh là polyp thì Dice của nó bằng
`2f / (1 + f)` với `f` là tỉ lệ pixel polyp. Đây là **sàn** của bài toán:
mọi kết quả dưới mức này còn tệ hơn việc không học gì.

Con số này đặt lại thang đo cho cả bảng kết quả, và giải thích vì sao mọi
đường val Dice đều xuất phát từ cùng một điểm ở epoch 1 (lúc đó sigmoid ra
~0.5 khắp nơi nên mô hình đoán tất cả là polyp).

In [ ]:
trivial_dice = 2 * ratio / (1 + ratio)
print(f"Dice của bộ dự đoán 'tất cả là polyp' : {trivial_dice:.4f}")
print(f"Mọi cấu hình phải vượt rõ mức này mới coi là có học được gì.")

## 4. Xem thử vài mẫu

Kiểm mắt thường rằng ảnh và mask khớp nhau.

In [ ]:
import matplotlib.pyplot as plt

ds = KvasirSegDataset(cfg.data_root, splits["train"][:6],
                      SegTransform(cfg.image_size, train=False))
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(6):
    img, msk = ds[i]
    axes[0, i].imshow(denormalize(img)); axes[0, i].axis("off")
    axes[1, i].imshow(msk[0], cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_title("ảnh", loc="left"); axes[1, 0].set_title("mask", loc="left")
plt.tight_layout()

## 5. Kiểm chứng mask đã nhị phân

Mask Kvasir lưu dạng JPEG nên có nhiễu nén. Sau khi qua `SegTransform`,
tensor mask chỉ được chứa đúng hai giá trị 0 và 1.

In [ ]:
import torch
vals = torch.unique(torch.cat([ds[i][1].flatten() for i in range(6)]))
print("Các giá trị có trong mask:", vals.tolist())
assert set(vals.tolist()) <= {0.0, 1.0}, "Mask chưa được nhị phân hoá!"
print("Mask đã nhị phân đúng.")